# 1. The network, the simulator, and the core idea

This notebook shows what the data *is*: a generated multi-echelon supply
network, a mechanistic simulator over it, and the one structural fact that
a per-supplier risk table cannot express.

**The claim being illustrated:** risk is a property of a node's position,
not of its attributes.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "2")
import sys
sys.path.insert(0, "../src")
import numpy as np, pandas as pd, torch
torch.set_num_threads(2)
import matplotlib.pyplot as plt
pd.set_option("display.width", 130)


In [ ]:
from sndsur.data.network import NetworkSpec, generate_network, N_TIERS
from sndsur.config import load_config
cfg = load_config('../configs/base.yaml')
from sndsur.data.scenarios import _net_spec, _sim_config
spec, sim = _net_spec(cfg), _sim_config(cfg)
net = generate_network(spec, seed=0, name='demo')
net.compute_throughput()
print(f'{net.n_nodes} nodes, {len(net.edges())} edges, {net.demand_nodes.size} demand points, {net.n_regions} regions')
groups = [g for gs in net.groups for g in gs]
print(f'{len(groups)} input groups, {sum(g.sole_source for g in groups)/len(groups):.0%} sole-sourced')

## The network, drawn by echelon

Node size is expected throughput. **Red edges are sole-source links** — the
only member of a customer's input group. Those are the hard single points of
failure, and nothing about the supplier's own attributes reveals them.

In [ ]:
from sndsur.viz import figure_network_example
import os
os.chdir('..')
p = figure_network_example()
os.chdir('notebooks')
from IPython.display import Image, display
display(Image(filename='../' + str(p)))

## Attributes versus position

Below, every node's *own* attributes sit next to its **sole-source reach** —
the number of demand points it can starve outright. The two are only weakly
related, which is the whole problem with scoring suppliers on features.

In [ ]:
import pandas as pd, numpy as np
thr = net.throughput
slack = np.where(np.isfinite(net.capacity), net.capacity/np.maximum(thr,1e-9)-1, np.nan)
df = pd.DataFrame({
    'node': np.arange(net.n_nodes), 'tier': net.tier,
    'throughput': thr.round(1), 'capacity_slack': slack.round(2),
    'n_customers': [len(c) for c in net.customers()],
    'downstream_reach': net.downstream_reach().astype(int),
    'sole_source_reach': net.sole_source_reach().astype(int),
})
prod = df[df.tier < N_TIERS-1]
print('correlation of each attribute with sole-source reach:')
for c in ('throughput','capacity_slack','n_customers'):
    print(f'  {c:18s} {prod[c].corr(prod.sole_source_reach):+.3f}')
prod.sort_values('sole_source_reach', ascending=False).head(10)

## The simulator, and a hand-checkable case

A single chain with no inventory: an outage at the raw supplier must produce a
shortage after exactly the cumulative lead time, and it must be the entire
demand. This is the arithmetic the simulator is tested against.

In [ ]:
from sndsur.sim.simulator import SimConfig, simulate, counterfactual, demand_realisation
from sndsur.data.disruptions import Disruption, DisruptionSet
import sys; sys.path.insert(0, '../tests')
from conftest import make_chain
chain = make_chain(n_tiers=5, lead=1, base_stock=0., input_cover=0., cv=0.)
c = SimConfig(warmup=30, horizon=12, backlog=False)
print('baseline fill rate:', simulate(chain, c, demand_realisation(chain, c, 0), None).fill_rate)
ds = DisruptionSet([Disruption('supplier_outage', 0, -1, 0, 12, 1.0)]).bind(chain)
imp = counterfactual(chain, c, ds, 0)
print('excess unmet per period:', imp.unmet_traj[0].round(2))
print('time to impact:', imp.time_to_impact[0], '(4 legs x lead 1 = 4)')

## What a disruption actually does on a real network

The trajectory is delayed, then amplified, then decays. That shape is what the
surrogate has to learn — and it is why the trajectory decoder is a GRU rather
than 12 independent linear outputs.

In [ ]:
d = demand_realisation(net, sim, 3)
base = simulate(net, sim, d, None)
print(f'undisrupted fill rate {np.nanmean(base.fill_rate):.4f}')
ss = net.sole_source_reach(); target = int(np.argmax(ss))
ds = DisruptionSet([Disruption('supplier_outage', target, -1, 4, 10, 1.0)]).bind(net)
imp = counterfactual(net, sim, ds, 3, baseline=base)
print(f'outage at node {target} (sole-source reach {ss[target]:.0f}): total service loss {imp.total_service_loss:.4f}')
fig, ax = plt.subplots(figsize=(8,3.5))
for i, v in enumerate(net.demand_nodes):
    if imp.unmet_traj[i].sum() > 1e-6:
        ax.plot(imp.unmet_traj[i], label=f'demand {v}')
ax.axvspan(4, 14, alpha=.12, color='red', label='outage window')
ax.set_xlabel('period'); ax.set_ylabel('excess unmet units'); ax.legend(fontsize=8)
ax.set_title('Shortage propagates with a delay, then recovers'); plt.show()

## The cost of the oracle

Exhaustive counterfactual analysis means one simulator run per candidate. That
is exact and it is the thing the surrogate is trying to replace.

In [ ]:
import time
from sndsur.sim.simulator import exhaustive_node_criticality
t0 = time.perf_counter()
cands, scores = exhaustive_node_criticality(net, sim, 3, duration=8, start=4)
el = time.perf_counter()-t0
print(f'{cands.size} candidates in {el:.2f}s ({el/cands.size*1000:.1f} ms each)')
order = np.argsort(-scores)[:8]
pd.DataFrame({'node': cands[order], 'tier': net.tier[cands[order]],
              'true_service_loss': scores[order].round(4),
              'sole_source_reach': ss[cands[order]].astype(int)})